## load libraries

In [1]:
import numpy as np
import os

import torch.nn as nn
import torch
import torch.optim as optim

from sklearn.model_selection import train_test_split

from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import accuracy_score


from CNN_blocks import *

In [2]:
device="cpu"

## load data

In [3]:
signals=np.load("../dataset/signals.npy")
labels= np.load("../dataset/labels.npy")

In [4]:
print(signals.shape)
print(labels.shape)

(10, 12, 1000)
(10, 8)


In [5]:
x=signals
y=labels

x_train, x_test, y_train, y_test = train_test_split(x, y)

mean=x_train.mean(axis=(0,2), keepdims=True)
std = x_test.std(axis=(0,2), keepdims=True)

x_train=(x_train-mean)/(std + 1e-8)
x_test =(x_test -mean)/(std + 1e-8)

x_train=torch.tensor(x_train, dtype=torch.float)
y_train=torch.tensor(y_train, dtype=torch.long)
x_test=torch.tensor(x_test, dtype=torch.float)
y_test=torch.tensor(y_test, dtype=torch.long)

In [22]:
#delete this block
tmp=x_train[-3]

In [7]:
train_dataset=TensorDataset(x_train, y_train)

In [8]:
train_loader=DataLoader(
    train_dataset,
    batch_size=5,
    shuffle=True
)

In [9]:
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.CNN_blocks=CNN_blocks()
        
        self.network=nn.Sequential(
            nn.Linear(128000, 256),
            nn.ReLU(),
            
            nn.Linear(256, 80),
            nn.ReLU(),
            
            nn.Linear(80, 8)
        )

    def forward(self, x):
        cnn_output=self.CNN_blocks(x)
        net_in=cnn_output.flatten(1)
        return self.network(net_in)

In [10]:
model=Classifier().to(device)

In [11]:
criterian=nn.BCEWithLogitsLoss()

optimizer=optim.Adam(
    model.parameters(),
    lr=0.001
)

In [12]:
epochs=40
for epoch in range(epochs):
    model.train()
    total_loss=0
    
    for x_batch, y_batch in train_loader:
        x_batch=x_batch.to(device)
        y_batch=y_batch.to(device)

        output=model(x_batch)

        loss=criterian(output, y_batch.float())

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss+=loss.item()

        print(
            "epoch:  ", epoch, "/", epochs,"\n",
            "loss:   ", total_loss, "\n\n"
        )

/home/rahaan/Desktop/projects/GESH-ECG-anomaly-deltector/detector_model/VE/lib/python3.10/site-packages/torch/cuda/__init__.py:422: UserWarning: Found GPU0 NVIDIA GeForce MX330 which is of compute capability (CC) 6.1.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Your installed torch==2.13.0+cu130 does not include kernels for this GPU. Reinstall the same version against a CUDA build that does, e.g.:
  For CUDA 12.6 use pip install torch==2.13.0 --index-url https://download.pytorch.org/whl/cu126
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/rahaan/Desktop/projects/GESH-ECG-anomaly-deltector/detector_model/VE/lib/py

epoch:   0 / 40 
 loss:    0.6882632970809937 


epoch:   0 / 40 
 loss:    1.5100822448730469 


epoch:   1 / 40 
 loss:    1.1362509727478027 


epoch:   1 / 40 
 loss:    1.1495234966278076 


epoch:   2 / 40 
 loss:    0.2816043496131897 


epoch:   2 / 40 
 loss:    2.4097076058387756 


epoch:   3 / 40 
 loss:    0.10086965560913086 


epoch:   3 / 40 
 loss:    0.10086965560913086 


epoch:   4 / 40 
 loss:    3.242567572669941e-06 


epoch:   4 / 40 
 loss:    8.726194892005879e-06 


epoch:   5 / 40 
 loss:    2.8623691150642117e-07 


epoch:   5 / 40 
 loss:    1.8882886081428296e-05 


epoch:   6 / 40 
 loss:    1.4365650713443756e-05 


epoch:   6 / 40 
 loss:    1.4484860002994537e-05 


epoch:   7 / 40 
 loss:    6.264448165893555e-05 


epoch:   7 / 40 
 loss:    6.568805974893621e-05 


epoch:   8 / 40 
 loss:    1.5329333109548315e-05 


epoch:   8 / 40 
 loss:    2.069375113933347e-05 


epoch:   9 / 40 
 loss:    0.00011650323722278699 


epoch:   9 / 40 
 loss:    0

In [13]:
x_test=x_test.to(device)
model.eval()
with torch.no_grad():
    y_pred=model(x_test)
print(outputs)
y_pred=torch.argmax(outputs, dim=1)
print(y_pred)

NameError: name 'outputs' is not defined

In [16]:
y_test

tensor([[0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0]])

In [17]:
y_train

tensor([[0, 0, 0, 0, 1, 1, 1, 0],
        [0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 1, 0]])

In [22]:
x_train[-1]

tensor([[ 0.0781,  0.1438,  0.1701,  ...,  0.3309,  0.2886,  0.2402],
        [ 0.1704,  0.1312,  0.0746,  ...,  0.4535,  0.3651,  0.2897],
        [ 0.2398,  0.0923, -0.0452,  ...,  0.5069,  0.3833,  0.2905],
        ...,
        [ 0.1779,  0.0952, -0.0170,  ...,  0.5469,  0.4265,  0.3408],
        [ 0.0618, -0.0259, -0.1473,  ...,  0.5506,  0.4237,  0.3361],
        [ 0.2094,  0.1372,  0.0403,  ...,  0.1904,  0.1159,  0.0802]])

In [24]:
tmp=tmp.to(device)
tmp = tmp.unsqueeze(0)
model.eval()
with torch.no_grad():
    y_pred=model(tmp)
print(y_pred)

tensor([[-111.3179,  -28.9477, -100.2363,  -68.4964,   81.9889,   86.5339,
           24.6253, -123.3767]])


In [20]:
y_train

tensor([[0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 1, 1, 0],
        [0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0]])

In [21]:
y_test

tensor([[0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 1, 0],
        [0, 0, 0, 0, 1, 0, 0, 0]])

In [23]:
y_train[-3]

tensor([0, 0, 0, 0, 1, 1, 1, 0])